# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Uman-66/Flyrank-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [26]:
!pip -q install duckdb


In [27]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN is None:
    raise ValueError("HF_TOKEN was not found in Colab Secrets.")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

print("Hugging Face connection ready.")

Hugging Face connection ready.


In [28]:
DAILY_PATH = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/**/*.parquet"
)

con.sql(f"""
SELECT COUNT(*) AS total_rows
FROM read_parquet(
    '{DAILY_PATH}',
    hive_partitioning=true
)
""")

MONTH = "2026-03"

Section 1 — Unit of analysis + time window

Markdown:

One row represents the daily observed performance of one content item, for one client, on one report date. The grain is report_date + client_hash_id + content_hash_id.

Time window: month = '2026-03' (March 2026), a mid-panel month. June 2026 is the sealed final month and is not used to define label logic.

Target/label: the schema has no pre-built trend column, so the label is derived: for each (client, content) pair, trend_direction = 'down' if that row's gsc_clicks is below the pair's own trailing 7-observation average clicks (computed from strictly prior rows only), else 'up'. This is a self-referential, backward-looking comparison — no future data is used to build the label.

Purpose: refresh prioritization / content opportunity scoring. This is an observed/directional ranking task, not a causal claim.

In [29]:
con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT report_date) AS distinct_report_dates,
    MIN(report_date) AS first_report_date,
    MAX(report_date) AS last_report_date
FROM read_parquet(
    '{DAILY_PATH}',
    hive_partitioning=true
)
WHERE month = '{MONTH}'
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────┬───────────────────────┬───────────────────┬──────────────────┐
│ row_count │ distinct_report_dates │ first_report_date │ last_report_date │
│   int64   │         int64         │       date        │       date       │
├───────────┼───────────────────────┼───────────────────┼──────────────────┤
│   9841378 │                    31 │ 2026-03-01        │ 2026-03-31       │
└───────────┴───────────────────────┴───────────────────┴──────────────────┘

In [30]:
con.sql(f"""
CREATE OR REPLACE VIEW content_daily_labeled AS
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    client_has_gsc,
    gsc_data_available,
    gsc_clicks,
    gsc_impressions,
    gsc_avg_position,
    ga4_engaged_sessions,

    COUNT(gsc_clicks) OVER (
        PARTITION BY client_hash_id, content_hash_id
        ORDER BY report_date
        ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
    ) AS prior_obs_count,

    AVG(gsc_clicks) OVER (
        PARTITION BY client_hash_id, content_hash_id
        ORDER BY report_date
        ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
    ) AS trailing_avg_clicks_7d

FROM read_parquet(
    '{DAILY_PATH}',
    hive_partitioning=true
)
WHERE month = '{MONTH}'
""")

print("view created")

view created


In [31]:
con.sql("""
SELECT
    *,
    CASE
        WHEN prior_obs_count >= 1 AND gsc_clicks < trailing_avg_clicks_7d THEN 'down'
        WHEN prior_obs_count >= 1 THEN 'up'
        ELSE NULL
    END AS trend_direction
FROM content_daily_labeled
LIMIT 5
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────────┬────────────┬─────────────────┬──────────────────┬──────────────────────┬─────────────────┬────────────────────────┬─────────────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ gsc_data_available │ gsc_clicks │ gsc_impressions │ gsc_avg_position │ ga4_engaged_sessions │ prior_obs_count │ trailing_avg_clicks_7d │ trend_direction │
│    date     │         varchar         │         varchar          │    boolean     │      boolean       │   int64    │      int64      │      double      │        int64         │      int64      │         double         │     varchar     │
├─────────────┼─────────────────────────┼──────────────────────────┼────────────────┼────────────────────┼────────────┼─────────────────┼──────────────────┼──────────────────────┼─────────────────┼────────────────────────┼─────────────────┤
│ 2026-03-01  │ client_0797ff3a1fc9a

Section 2 — Fields: feature / label / context / excluded

Markdown:

Label: trend_direction (derived, see Section 1) — down means the day's clicks fell below the content's own recent trailing average. This is the outcome and is never used as an input feature.

Context: client_hash_id, content_hash_id, report_date, month.

Features (five):

trailing_avg_clicks_7d — knowable at the decision moment because it's computed only from the 7 prior observations of the same content, excluding the current day.
gsc_avg_position — knowable at the decision moment because it's the search ranking position observed as of that day's report, not a future value.
gsc_impressions — knowable at the decision moment because it's the same-day observed impression count, already measured by the time the report exists.
ga4_engaged_sessions — knowable at the decision moment because it's the same-day observed engagement count.
client_has_gsc — knowable at the decision moment because it's a static client attribute, true regardless of date.

Excluded: trend_direction (the label itself), prior_obs_count (a bookkeeping column, not a real signal), and anything computed from a period after the current report_date — none of the five features above reach into the future, but any such column would be excluded for that reason.

In [32]:
con.sql("""
SELECT
    CASE
        WHEN prior_obs_count >= 1 AND gsc_clicks < trailing_avg_clicks_7d THEN 'down'
        WHEN prior_obs_count >= 1 THEN 'up'
        ELSE NULL
    END AS trend_direction,
    COUNT(*) AS rows
FROM content_daily_labeled
GROUP BY 1
ORDER BY rows DESC
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┬─────────┐
│ trend_direction │  rows   │
│     varchar     │  int64  │
├─────────────────┼─────────┤
│ up              │ 8690804 │
│ down            │  819137 │
│ NULL            │  331437 │
└─────────────────┴─────────┘

In [33]:
con.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE client_hash_id IS NULL) AS missing_client_hash_id,
    COUNT(*) FILTER (WHERE content_hash_id IS NULL) AS missing_content_hash_id,
    COUNT(*) FILTER (WHERE report_date IS NULL) AS missing_report_date,
    COUNT(*) FILTER (WHERE prior_obs_count = 0) AS rows_with_no_prior_history
FROM content_daily_labeled
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────────┬─────────────────────────┬─────────────────────┬────────────────────────────┐
│ total_rows │ missing_client_hash_id │ missing_content_hash_id │ missing_report_date │ rows_with_no_prior_history │
│   int64    │         int64          │          int64          │        int64        │           int64            │
├────────────┼────────────────────────┼─────────────────────────┼─────────────────────┼────────────────────────────┤
│    9841378 │                      0 │                       0 │                   0 │                     331437 │
└────────────┴────────────────────────┴─────────────────────────┴─────────────────────┴────────────────────────────┘

Section 3 — Verify it with queries

Markdown — grain verification:

Expected grain: one row per report_date + client_hash_id + content_hash_id.

In [34]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS rows_per_grain
FROM read_parquet(
    '{DAILY_PATH}',
    hive_partitioning=true
)
WHERE month = '{MONTH}'
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
ORDER BY rows_per_grain DESC
LIMIT 20
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬────────────────┐
│ report_date │ client_hash_id │ content_hash_id │ rows_per_grain │
│    date     │    varchar     │     varchar     │     int64      │
├─────────────┴────────────────┴─────────────────┴────────────────┤
│                             0 rows                              │
└─────────────────────────────────────────────────────────────────┘

In [35]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT
        CAST(report_date AS VARCHAR) || '|' ||
        CAST(client_hash_id AS VARCHAR) || '|' ||
        CAST(content_hash_id AS VARCHAR)
    ) AS unique_grain_keys
FROM read_parquet(
    '{DAILY_PATH}',
    hive_partitioning=true
)
WHERE month = '{MONTH}'
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬───────────────────┐
│ total_rows │ unique_grain_keys │
│   int64    │       int64       │
├────────────┼───────────────────┤
│    9841378 │           9841378 │
└────────────┴───────────────────┘

In [36]:
con.sql(f"""
SELECT
    COUNT(*) AS all_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS rows_surviving_is_true
FROM read_parquet(
    '{DAILY_PATH}',
    hive_partitioning=true
)
WHERE month = '{MONTH}'
""")

┌──────────┬────────────────────────┐
│ all_rows │ rows_surviving_is_true │
│  int64   │         int64          │
├──────────┼────────────────────────┤
│  9841378 │                3611061 │
└──────────┴────────────────────────┘

In [37]:
con.sql("""
WITH labeled AS (
    SELECT
        *,
        CASE
            WHEN prior_obs_count >= 1 AND gsc_clicks < trailing_avg_clicks_7d THEN 'down'
            WHEN prior_obs_count >= 1 THEN 'up'
            ELSE NULL
        END AS trend_direction
    FROM content_daily_labeled
)
SELECT
    trend_direction,
    CASE WHEN trend_direction = 'down' THEN 1 ELSE 0 END AS leaked_decline_feature
FROM labeled
LIMIT 20
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┬────────────────────────┐
│ trend_direction │ leaked_decline_feature │
│     varchar     │         int32          │
├─────────────────┼────────────────────────┤
│ NULL            │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up      

In [38]:
con.sql("""
WITH labeled AS (
    SELECT
        *,
        CASE
            WHEN prior_obs_count >= 1 AND gsc_clicks < trailing_avg_clicks_7d THEN 'down'
            WHEN prior_obs_count >= 1 THEN 'up'
            ELSE NULL
        END AS trend_direction
    FROM content_daily_labeled
),
checked AS (
    SELECT
        trend_direction,
        CASE WHEN trend_direction = 'down' THEN 1 ELSE 0 END AS leaked_decline_feature
    FROM labeled
)
SELECT
    COUNT(*) AS rows,
    SUM(CASE
        WHEN leaked_decline_feature = CASE WHEN trend_direction = 'down' THEN 1 ELSE 0 END
        THEN 1 ELSE 0
    END) AS matching_rows
FROM checked
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬───────────────┐
│  rows   │ matching_rows │
│  int64  │    int128     │
├─────────┼───────────────┤
│ 9841378 │       9841378 │
└─────────┴───────────────┘

Section 4 — Data limits

Markdown:

Unbalanced history. Content with few prior observations gets a noisier trailing_avg_clicks_7d baseline than long-running content — check rows_with_no_prior_history above.

Observational, not causal. A down label means clicks fell relative to the content's own recent baseline — it does not prove refreshing the page would reverse that.

GSC-only early rows. Some rows may have gsc_data_available = FALSE, meaning search-console-based label/features aren't trustworthy for that row (see the IS TRUE check above).

Window overlap / early-March edge effect. Because the label window only looks at prior rows within March, the first few days of March have a smaller trailing window than later days — a boundary effect, not a leak, but worth naming.

Sealed final month. June 2026 is not used to define label logic here.

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Uman-66/Flyrank-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
!pip -q install duckdb


In [ ]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN is None:
    raise ValueError("HF_TOKEN was not found in Colab Secrets.")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

print("Hugging Face connection ready.")

Hugging Face connection ready.


In [ ]:
DAILY_PATH = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/**/*.parquet"
)

con.sql(f"""
SELECT COUNT(*) AS total_rows
FROM read_parquet(
    '{DAILY_PATH}',
    hive_partitioning=true
)
""")

MONTH = "2026-03"

Section 1 — Unit of analysis + time window

Markdown:

One row represents the daily observed performance of one content item, for one client, on one report date. The grain is report_date + client_hash_id + content_hash_id.

Time window: month = '2026-03' (March 2026), a mid-panel month. June 2026 is the sealed final month and is not used to define label logic.

Target/label: the schema has no pre-built trend column, so the label is derived: for each (client, content) pair, trend_direction = 'down' if that row's gsc_clicks is below the pair's own trailing 7-observation average clicks (computed from strictly prior rows only), else 'up'. This is a self-referential, backward-looking comparison — no future data is used to build the label.

Purpose: refresh prioritization / content opportunity scoring. This is an observed/directional ranking task, not a causal claim.

In [ ]:
con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT report_date) AS distinct_report_dates,
    MIN(report_date) AS first_report_date,
    MAX(report_date) AS last_report_date
FROM read_parquet(
    '{DAILY_PATH}',
    hive_partitioning=true
)
WHERE month = '{MONTH}'
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────┬───────────────────────┬───────────────────┬──────────────────┐
│ row_count │ distinct_report_dates │ first_report_date │ last_report_date │
│   int64   │         int64         │       date        │       date       │
├───────────┼───────────────────────┼───────────────────┼──────────────────┤
│   9841378 │                    31 │ 2026-03-01        │ 2026-03-31       │
└───────────┴───────────────────────┴───────────────────┴──────────────────┘

In [ ]:
con.sql(f"""
CREATE OR REPLACE VIEW content_daily_labeled AS
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    client_has_gsc,
    gsc_data_available,
    gsc_clicks,
    gsc_impressions,
    gsc_avg_position,
    ga4_engaged_sessions,

    COUNT(gsc_clicks) OVER (
        PARTITION BY client_hash_id, content_hash_id
        ORDER BY report_date
        ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
    ) AS prior_obs_count,

    AVG(gsc_clicks) OVER (
        PARTITION BY client_hash_id, content_hash_id
        ORDER BY report_date
        ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
    ) AS trailing_avg_clicks_7d

FROM read_parquet(
    '{DAILY_PATH}',
    hive_partitioning=true
)
WHERE month = '{MONTH}'
""")

print("view created")

view created


In [ ]:
con.sql("""
SELECT
    *,
    CASE
        WHEN prior_obs_count >= 1 AND gsc_clicks < trailing_avg_clicks_7d THEN 'down'
        WHEN prior_obs_count >= 1 THEN 'up'
        ELSE NULL
    END AS trend_direction
FROM content_daily_labeled
LIMIT 5
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────────┬────────────┬─────────────────┬──────────────────┬──────────────────────┬─────────────────┬────────────────────────┬─────────────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ gsc_data_available │ gsc_clicks │ gsc_impressions │ gsc_avg_position │ ga4_engaged_sessions │ prior_obs_count │ trailing_avg_clicks_7d │ trend_direction │
│    date     │         varchar         │         varchar          │    boolean     │      boolean       │   int64    │      int64      │      double      │        int64         │      int64      │         double         │     varchar     │
├─────────────┼─────────────────────────┼──────────────────────────┼────────────────┼────────────────────┼────────────┼─────────────────┼──────────────────┼──────────────────────┼─────────────────┼────────────────────────┼─────────────────┤
│ 2026-03-01  │ client_0797ff3a1fc9a

Section 2 — Fields: feature / label / context / excluded

Markdown:

Label: trend_direction (derived, see Section 1) — down means the day's clicks fell below the content's own recent trailing average. This is the outcome and is never used as an input feature.

Context: client_hash_id, content_hash_id, report_date, month.

Features (five):

trailing_avg_clicks_7d — knowable at the decision moment because it's computed only from the 7 prior observations of the same content, excluding the current day.
gsc_avg_position — knowable at the decision moment because it's the search ranking position observed as of that day's report, not a future value.
gsc_impressions — knowable at the decision moment because it's the same-day observed impression count, already measured by the time the report exists.
ga4_engaged_sessions — knowable at the decision moment because it's the same-day observed engagement count.
client_has_gsc — knowable at the decision moment because it's a static client attribute, true regardless of date.

Excluded: trend_direction (the label itself), prior_obs_count (a bookkeeping column, not a real signal), and anything computed from a period after the current report_date — none of the five features above reach into the future, but any such column would be excluded for that reason.

In [ ]:
con.sql("""
SELECT
    CASE
        WHEN prior_obs_count >= 1 AND gsc_clicks < trailing_avg_clicks_7d THEN 'down'
        WHEN prior_obs_count >= 1 THEN 'up'
        ELSE NULL
    END AS trend_direction,
    COUNT(*) AS rows
FROM content_daily_labeled
GROUP BY 1
ORDER BY rows DESC
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┬─────────┐
│ trend_direction │  rows   │
│     varchar     │  int64  │
├─────────────────┼─────────┤
│ up              │ 8690804 │
│ down            │  819137 │
│ NULL            │  331437 │
└─────────────────┴─────────┘

In [ ]:
con.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE client_hash_id IS NULL) AS missing_client_hash_id,
    COUNT(*) FILTER (WHERE content_hash_id IS NULL) AS missing_content_hash_id,
    COUNT(*) FILTER (WHERE report_date IS NULL) AS missing_report_date,
    COUNT(*) FILTER (WHERE prior_obs_count = 0) AS rows_with_no_prior_history
FROM content_daily_labeled
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────────┬─────────────────────────┬─────────────────────┬────────────────────────────┐
│ total_rows │ missing_client_hash_id │ missing_content_hash_id │ missing_report_date │ rows_with_no_prior_history │
│   int64    │         int64          │          int64          │        int64        │           int64            │
├────────────┼────────────────────────┼─────────────────────────┼─────────────────────┼────────────────────────────┤
│    9841378 │                      0 │                       0 │                   0 │                     331437 │
└────────────┴────────────────────────┴─────────────────────────┴─────────────────────┴────────────────────────────┘

Section 3 — Verify it with queries

Markdown — grain verification:

Expected grain: one row per report_date + client_hash_id + content_hash_id.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS rows_per_grain
FROM read_parquet(
    '{DAILY_PATH}',
    hive_partitioning=true
)
WHERE month = '{MONTH}'
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
ORDER BY rows_per_grain DESC
LIMIT 20
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬────────────────┐
│ report_date │ client_hash_id │ content_hash_id │ rows_per_grain │
│    date     │    varchar     │     varchar     │     int64      │
├─────────────┴────────────────┴─────────────────┴────────────────┤
│                             0 rows                              │
└─────────────────────────────────────────────────────────────────┘

In [ ]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT
        CAST(report_date AS VARCHAR) || '|' ||
        CAST(client_hash_id AS VARCHAR) || '|' ||
        CAST(content_hash_id AS VARCHAR)
    ) AS unique_grain_keys
FROM read_parquet(
    '{DAILY_PATH}',
    hive_partitioning=true
)
WHERE month = '{MONTH}'
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬───────────────────┐
│ total_rows │ unique_grain_keys │
│   int64    │       int64       │
├────────────┼───────────────────┤
│    9841378 │           9841378 │
└────────────┴───────────────────┘

In [ ]:
con.sql(f"""
SELECT
    COUNT(*) AS all_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS rows_surviving_is_true
FROM read_parquet(
    '{DAILY_PATH}',
    hive_partitioning=true
)
WHERE month = '{MONTH}'
""")

┌──────────┬────────────────────────┐
│ all_rows │ rows_surviving_is_true │
│  int64   │         int64          │
├──────────┼────────────────────────┤
│  9841378 │                3611061 │
└──────────┴────────────────────────┘

In [ ]:
con.sql("""
WITH labeled AS (
    SELECT
        *,
        CASE
            WHEN prior_obs_count >= 1 AND gsc_clicks < trailing_avg_clicks_7d THEN 'down'
            WHEN prior_obs_count >= 1 THEN 'up'
            ELSE NULL
        END AS trend_direction
    FROM content_daily_labeled
)
SELECT
    trend_direction,
    CASE WHEN trend_direction = 'down' THEN 1 ELSE 0 END AS leaked_decline_feature
FROM labeled
LIMIT 20
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┬────────────────────────┐
│ trend_direction │ leaked_decline_feature │
│     varchar     │         int32          │
├─────────────────┼────────────────────────┤
│ NULL            │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up      

In [ ]:
con.sql("""
WITH labeled AS (
    SELECT
        *,
        CASE
            WHEN prior_obs_count >= 1 AND gsc_clicks < trailing_avg_clicks_7d THEN 'down'
            WHEN prior_obs_count >= 1 THEN 'up'
            ELSE NULL
        END AS trend_direction
    FROM content_daily_labeled
),
checked AS (
    SELECT
        trend_direction,
        CASE WHEN trend_direction = 'down' THEN 1 ELSE 0 END AS leaked_decline_feature
    FROM labeled
)
SELECT
    COUNT(*) AS rows,
    SUM(CASE
        WHEN leaked_decline_feature = CASE WHEN trend_direction = 'down' THEN 1 ELSE 0 END
        THEN 1 ELSE 0
    END) AS matching_rows
FROM checked
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬───────────────┐
│  rows   │ matching_rows │
│  int64  │    int128     │
├─────────┼───────────────┤
│ 9841378 │       9841378 │
└─────────┴───────────────┘

Section 4 — Data limits

Markdown:

Unbalanced history. Content with few prior observations gets a noisier trailing_avg_clicks_7d baseline than long-running content — check rows_with_no_prior_history above.

Observational, not causal. A down label means clicks fell relative to the content's own recent baseline — it does not prove refreshing the page would reverse that.

GSC-only early rows. Some rows may have gsc_data_available = FALSE, meaning search-console-based label/features aren't trustworthy for that row (see the IS TRUE check above).

Window overlap / early-March edge effect. Because the label window only looks at prior rows within March, the first few days of March have a smaller trailing window than later days — a boundary effect, not a leak, but worth naming.

Sealed final month. June 2026 is not used to define label logic here.

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Uman-66/Flyrank-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
!pip -q install duckdb


In [ ]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN is None:
    raise ValueError("HF_TOKEN was not found in Colab Secrets.")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

print("Hugging Face connection ready.")

Hugging Face connection ready.


In [ ]:
DAILY_PATH = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/**/*.parquet"
)

con.sql(f"""
SELECT COUNT(*) AS total_rows
FROM read_parquet(
    '{DAILY_PATH}',
    hive_partitioning=true
)
""")

MONTH = "2026-03"

Section 1 — Unit of analysis + time window

Markdown:

One row represents the daily observed performance of one content item, for one client, on one report date. The grain is report_date + client_hash_id + content_hash_id.

Time window: month = '2026-03' (March 2026), a mid-panel month. June 2026 is the sealed final month and is not used to define label logic.

Target/label: the schema has no pre-built trend column, so the label is derived: for each (client, content) pair, trend_direction = 'down' if that row's gsc_clicks is below the pair's own trailing 7-observation average clicks (computed from strictly prior rows only), else 'up'. This is a self-referential, backward-looking comparison — no future data is used to build the label.

Purpose: refresh prioritization / content opportunity scoring. This is an observed/directional ranking task, not a causal claim.

In [ ]:
con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT report_date) AS distinct_report_dates,
    MIN(report_date) AS first_report_date,
    MAX(report_date) AS last_report_date
FROM read_parquet(
    '{DAILY_PATH}',
    hive_partitioning=true
)
WHERE month = '{MONTH}'
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────┬───────────────────────┬───────────────────┬──────────────────┐
│ row_count │ distinct_report_dates │ first_report_date │ last_report_date │
│   int64   │         int64         │       date        │       date       │
├───────────┼───────────────────────┼───────────────────┼──────────────────┤
│   9841378 │                    31 │ 2026-03-01        │ 2026-03-31       │
└───────────┴───────────────────────┴───────────────────┴──────────────────┘

In [ ]:
con.sql(f"""
CREATE OR REPLACE VIEW content_daily_labeled AS
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    client_has_gsc,
    gsc_data_available,
    gsc_clicks,
    gsc_impressions,
    gsc_avg_position,
    ga4_engaged_sessions,

    COUNT(gsc_clicks) OVER (
        PARTITION BY client_hash_id, content_hash_id
        ORDER BY report_date
        ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
    ) AS prior_obs_count,

    AVG(gsc_clicks) OVER (
        PARTITION BY client_hash_id, content_hash_id
        ORDER BY report_date
        ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
    ) AS trailing_avg_clicks_7d

FROM read_parquet(
    '{DAILY_PATH}',
    hive_partitioning=true
)
WHERE month = '{MONTH}'
""")

print("view created")

view created


In [ ]:
con.sql("""
SELECT
    *,
    CASE
        WHEN prior_obs_count >= 1 AND gsc_clicks < trailing_avg_clicks_7d THEN 'down'
        WHEN prior_obs_count >= 1 THEN 'up'
        ELSE NULL
    END AS trend_direction
FROM content_daily_labeled
LIMIT 5
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────────┬────────────┬─────────────────┬──────────────────┬──────────────────────┬─────────────────┬────────────────────────┬─────────────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ gsc_data_available │ gsc_clicks │ gsc_impressions │ gsc_avg_position │ ga4_engaged_sessions │ prior_obs_count │ trailing_avg_clicks_7d │ trend_direction │
│    date     │         varchar         │         varchar          │    boolean     │      boolean       │   int64    │      int64      │      double      │        int64         │      int64      │         double         │     varchar     │
├─────────────┼─────────────────────────┼──────────────────────────┼────────────────┼────────────────────┼────────────┼─────────────────┼──────────────────┼──────────────────────┼─────────────────┼────────────────────────┼─────────────────┤
│ 2026-03-01  │ client_0797ff3a1fc9a

Section 2 — Fields: feature / label / context / excluded

Markdown:

Label: trend_direction (derived, see Section 1) — down means the day's clicks fell below the content's own recent trailing average. This is the outcome and is never used as an input feature.

Context: client_hash_id, content_hash_id, report_date, month.

Features (five):

trailing_avg_clicks_7d — knowable at the decision moment because it's computed only from the 7 prior observations of the same content, excluding the current day.
gsc_avg_position — knowable at the decision moment because it's the search ranking position observed as of that day's report, not a future value.
gsc_impressions — knowable at the decision moment because it's the same-day observed impression count, already measured by the time the report exists.
ga4_engaged_sessions — knowable at the decision moment because it's the same-day observed engagement count.
client_has_gsc — knowable at the decision moment because it's a static client attribute, true regardless of date.

Excluded: trend_direction (the label itself), prior_obs_count (a bookkeeping column, not a real signal), and anything computed from a period after the current report_date — none of the five features above reach into the future, but any such column would be excluded for that reason.

In [ ]:
con.sql("""
SELECT
    CASE
        WHEN prior_obs_count >= 1 AND gsc_clicks < trailing_avg_clicks_7d THEN 'down'
        WHEN prior_obs_count >= 1 THEN 'up'
        ELSE NULL
    END AS trend_direction,
    COUNT(*) AS rows
FROM content_daily_labeled
GROUP BY 1
ORDER BY rows DESC
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┬─────────┐
│ trend_direction │  rows   │
│     varchar     │  int64  │
├─────────────────┼─────────┤
│ up              │ 8690804 │
│ down            │  819137 │
│ NULL            │  331437 │
└─────────────────┴─────────┘

In [ ]:
con.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE client_hash_id IS NULL) AS missing_client_hash_id,
    COUNT(*) FILTER (WHERE content_hash_id IS NULL) AS missing_content_hash_id,
    COUNT(*) FILTER (WHERE report_date IS NULL) AS missing_report_date,
    COUNT(*) FILTER (WHERE prior_obs_count = 0) AS rows_with_no_prior_history
FROM content_daily_labeled
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────────┬─────────────────────────┬─────────────────────┬────────────────────────────┐
│ total_rows │ missing_client_hash_id │ missing_content_hash_id │ missing_report_date │ rows_with_no_prior_history │
│   int64    │         int64          │          int64          │        int64        │           int64            │
├────────────┼────────────────────────┼─────────────────────────┼─────────────────────┼────────────────────────────┤
│    9841378 │                      0 │                       0 │                   0 │                     331437 │
└────────────┴────────────────────────┴─────────────────────────┴─────────────────────┴────────────────────────────┘

Section 3 — Verify it with queries

Markdown — grain verification:

Expected grain: one row per report_date + client_hash_id + content_hash_id.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS rows_per_grain
FROM read_parquet(
    '{DAILY_PATH}',
    hive_partitioning=true
)
WHERE month = '{MONTH}'
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
ORDER BY rows_per_grain DESC
LIMIT 20
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬────────────────┐
│ report_date │ client_hash_id │ content_hash_id │ rows_per_grain │
│    date     │    varchar     │     varchar     │     int64      │
├─────────────┴────────────────┴─────────────────┴────────────────┤
│                             0 rows                              │
└─────────────────────────────────────────────────────────────────┘

In [ ]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT
        CAST(report_date AS VARCHAR) || '|' ||
        CAST(client_hash_id AS VARCHAR) || '|' ||
        CAST(content_hash_id AS VARCHAR)
    ) AS unique_grain_keys
FROM read_parquet(
    '{DAILY_PATH}',
    hive_partitioning=true
)
WHERE month = '{MONTH}'
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬───────────────────┐
│ total_rows │ unique_grain_keys │
│   int64    │       int64       │
├────────────┼───────────────────┤
│    9841378 │           9841378 │
└────────────┴───────────────────┘

In [ ]:
con.sql(f"""
SELECT
    COUNT(*) AS all_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS rows_surviving_is_true
FROM read_parquet(
    '{DAILY_PATH}',
    hive_partitioning=true
)
WHERE month = '{MONTH}'
""")

┌──────────┬────────────────────────┐
│ all_rows │ rows_surviving_is_true │
│  int64   │         int64          │
├──────────┼────────────────────────┤
│  9841378 │                3611061 │
└──────────┴────────────────────────┘

In [ ]:
con.sql("""
WITH labeled AS (
    SELECT
        *,
        CASE
            WHEN prior_obs_count >= 1 AND gsc_clicks < trailing_avg_clicks_7d THEN 'down'
            WHEN prior_obs_count >= 1 THEN 'up'
            ELSE NULL
        END AS trend_direction
    FROM content_daily_labeled
)
SELECT
    trend_direction,
    CASE WHEN trend_direction = 'down' THEN 1 ELSE 0 END AS leaked_decline_feature
FROM labeled
LIMIT 20
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┬────────────────────────┐
│ trend_direction │ leaked_decline_feature │
│     varchar     │         int32          │
├─────────────────┼────────────────────────┤
│ NULL            │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up      

In [40]:
con.sql("""
WITH labeled AS (
    SELECT
        *,
        CASE
            WHEN prior_obs_count >= 1 AND gsc_clicks < trailing_avg_clicks_7d THEN 'down'
            WHEN prior_obs_count >= 1 THEN 'up'
            ELSE NULL
        END AS trend_direction
    FROM content_daily_labeled
),
checked AS (
    SELECT
        trend_direction,
        CASE WHEN trend_direction = 'down' THEN 1 ELSE 0 END AS leaked_decline_feature
    FROM labeled
)
SELECT
    COUNT(*) AS rows,
    SUM(CASE
        WHEN leaked_decline_feature = CASE WHEN trend_direction = 'down' THEN 1 ELSE 0 END
        THEN 1 ELSE 0
    END) AS matching_rows
FROM checked
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬───────────────┐
│  rows   │ matching_rows │
│  int64  │    int128     │
├─────────┼───────────────┤
│ 9841378 │       9841378 │
└─────────┴───────────────┘

Section 4 — Data limits

Markdown:

Unbalanced history. Content with few prior observations gets a noisier trailing_avg_clicks_7d baseline than long-running content — check rows_with_no_prior_history above.

Observational, not causal. A down label means clicks fell relative to the content's own recent baseline — it does not prove refreshing the page would reverse that.

GSC-only early rows. Some rows may have gsc_data_available = FALSE, meaning search-console-based label/features aren't trustworthy for that row (see the IS TRUE check above).

Window overlap / early-March edge effect. Because the label window only looks at prior rows within March, the first few days of March have a smaller trailing window than later days — a boundary effect, not a leak, but worth naming.

Sealed final month. June 2026 is not used to define label logic here.

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Uman-66/Flyrank-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
!pip -q install duckdb


In [ ]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN is None:
    raise ValueError("HF_TOKEN was not found in Colab Secrets.")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

print("Hugging Face connection ready.")

Hugging Face connection ready.


In [ ]:
DAILY_PATH = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/**/*.parquet"
)

con.sql(f"""
SELECT COUNT(*) AS total_rows
FROM read_parquet(
    '{DAILY_PATH}',
    hive_partitioning=true
)
""")

MONTH = "2026-03"

Section 1 — Unit of analysis + time window

Markdown:

One row represents the daily observed performance of one content item, for one client, on one report date. The grain is report_date + client_hash_id + content_hash_id.

Time window: month = '2026-03' (March 2026), a mid-panel month. June 2026 is the sealed final month and is not used to define label logic.

Target/label: the schema has no pre-built trend column, so the label is derived: for each (client, content) pair, trend_direction = 'down' if that row's gsc_clicks is below the pair's own trailing 7-observation average clicks (computed from strictly prior rows only), else 'up'. This is a self-referential, backward-looking comparison — no future data is used to build the label.

Purpose: refresh prioritization / content opportunity scoring. This is an observed/directional ranking task, not a causal claim.

In [ ]:
con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT report_date) AS distinct_report_dates,
    MIN(report_date) AS first_report_date,
    MAX(report_date) AS last_report_date
FROM read_parquet(
    '{DAILY_PATH}',
    hive_partitioning=true
)
WHERE month = '{MONTH}'
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────┬───────────────────────┬───────────────────┬──────────────────┐
│ row_count │ distinct_report_dates │ first_report_date │ last_report_date │
│   int64   │         int64         │       date        │       date       │
├───────────┼───────────────────────┼───────────────────┼──────────────────┤
│   9841378 │                    31 │ 2026-03-01        │ 2026-03-31       │
└───────────┴───────────────────────┴───────────────────┴──────────────────┘

In [ ]:
con.sql(f"""
CREATE OR REPLACE VIEW content_daily_labeled AS
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    client_has_gsc,
    gsc_data_available,
    gsc_clicks,
    gsc_impressions,
    gsc_avg_position,
    ga4_engaged_sessions,

    COUNT(gsc_clicks) OVER (
        PARTITION BY client_hash_id, content_hash_id
        ORDER BY report_date
        ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
    ) AS prior_obs_count,

    AVG(gsc_clicks) OVER (
        PARTITION BY client_hash_id, content_hash_id
        ORDER BY report_date
        ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
    ) AS trailing_avg_clicks_7d

FROM read_parquet(
    '{DAILY_PATH}',
    hive_partitioning=true
)
WHERE month = '{MONTH}'
""")

print("view created")

view created


In [ ]:
con.sql("""
SELECT
    *,
    CASE
        WHEN prior_obs_count >= 1 AND gsc_clicks < trailing_avg_clicks_7d THEN 'down'
        WHEN prior_obs_count >= 1 THEN 'up'
        ELSE NULL
    END AS trend_direction
FROM content_daily_labeled
LIMIT 5
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────────┬────────────┬─────────────────┬──────────────────┬──────────────────────┬─────────────────┬────────────────────────┬─────────────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ gsc_data_available │ gsc_clicks │ gsc_impressions │ gsc_avg_position │ ga4_engaged_sessions │ prior_obs_count │ trailing_avg_clicks_7d │ trend_direction │
│    date     │         varchar         │         varchar          │    boolean     │      boolean       │   int64    │      int64      │      double      │        int64         │      int64      │         double         │     varchar     │
├─────────────┼─────────────────────────┼──────────────────────────┼────────────────┼────────────────────┼────────────┼─────────────────┼──────────────────┼──────────────────────┼─────────────────┼────────────────────────┼─────────────────┤
│ 2026-03-01  │ client_0797ff3a1fc9a

Section 2 — Fields: feature / label / context / excluded

Markdown:

Label: trend_direction (derived, see Section 1) — down means the day's clicks fell below the content's own recent trailing average. This is the outcome and is never used as an input feature.

Context: client_hash_id, content_hash_id, report_date, month.

Features (five):

trailing_avg_clicks_7d — knowable at the decision moment because it's computed only from the 7 prior observations of the same content, excluding the current day.
gsc_avg_position — knowable at the decision moment because it's the search ranking position observed as of that day's report, not a future value.
gsc_impressions — knowable at the decision moment because it's the same-day observed impression count, already measured by the time the report exists.
ga4_engaged_sessions — knowable at the decision moment because it's the same-day observed engagement count.
client_has_gsc — knowable at the decision moment because it's a static client attribute, true regardless of date.

Excluded: trend_direction (the label itself), prior_obs_count (a bookkeeping column, not a real signal), and anything computed from a period after the current report_date — none of the five features above reach into the future, but any such column would be excluded for that reason.

In [ ]:
con.sql("""
SELECT
    CASE
        WHEN prior_obs_count >= 1 AND gsc_clicks < trailing_avg_clicks_7d THEN 'down'
        WHEN prior_obs_count >= 1 THEN 'up'
        ELSE NULL
    END AS trend_direction,
    COUNT(*) AS rows
FROM content_daily_labeled
GROUP BY 1
ORDER BY rows DESC
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┬─────────┐
│ trend_direction │  rows   │
│     varchar     │  int64  │
├─────────────────┼─────────┤
│ up              │ 8690804 │
│ down            │  819137 │
│ NULL            │  331437 │
└─────────────────┴─────────┘

In [ ]:
con.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE client_hash_id IS NULL) AS missing_client_hash_id,
    COUNT(*) FILTER (WHERE content_hash_id IS NULL) AS missing_content_hash_id,
    COUNT(*) FILTER (WHERE report_date IS NULL) AS missing_report_date,
    COUNT(*) FILTER (WHERE prior_obs_count = 0) AS rows_with_no_prior_history
FROM content_daily_labeled
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────────┬─────────────────────────┬─────────────────────┬────────────────────────────┐
│ total_rows │ missing_client_hash_id │ missing_content_hash_id │ missing_report_date │ rows_with_no_prior_history │
│   int64    │         int64          │          int64          │        int64        │           int64            │
├────────────┼────────────────────────┼─────────────────────────┼─────────────────────┼────────────────────────────┤
│    9841378 │                      0 │                       0 │                   0 │                     331437 │
└────────────┴────────────────────────┴─────────────────────────┴─────────────────────┴────────────────────────────┘

Section 3 — Verify it with queries

Markdown — grain verification:

Expected grain: one row per report_date + client_hash_id + content_hash_id.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS rows_per_grain
FROM read_parquet(
    '{DAILY_PATH}',
    hive_partitioning=true
)
WHERE month = '{MONTH}'
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
ORDER BY rows_per_grain DESC
LIMIT 20
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬────────────────┐
│ report_date │ client_hash_id │ content_hash_id │ rows_per_grain │
│    date     │    varchar     │     varchar     │     int64      │
├─────────────┴────────────────┴─────────────────┴────────────────┤
│                             0 rows                              │
└─────────────────────────────────────────────────────────────────┘

In [ ]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT
        CAST(report_date AS VARCHAR) || '|' ||
        CAST(client_hash_id AS VARCHAR) || '|' ||
        CAST(content_hash_id AS VARCHAR)
    ) AS unique_grain_keys
FROM read_parquet(
    '{DAILY_PATH}',
    hive_partitioning=true
)
WHERE month = '{MONTH}'
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬───────────────────┐
│ total_rows │ unique_grain_keys │
│   int64    │       int64       │
├────────────┼───────────────────┤
│    9841378 │           9841378 │
└────────────┴───────────────────┘

In [ ]:
con.sql(f"""
SELECT
    COUNT(*) AS all_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS rows_surviving_is_true
FROM read_parquet(
    '{DAILY_PATH}',
    hive_partitioning=true
)
WHERE month = '{MONTH}'
""")

┌──────────┬────────────────────────┐
│ all_rows │ rows_surviving_is_true │
│  int64   │         int64          │
├──────────┼────────────────────────┤
│  9841378 │                3611061 │
└──────────┴────────────────────────┘

In [ ]:
con.sql("""
WITH labeled AS (
    SELECT
        *,
        CASE
            WHEN prior_obs_count >= 1 AND gsc_clicks < trailing_avg_clicks_7d THEN 'down'
            WHEN prior_obs_count >= 1 THEN 'up'
            ELSE NULL
        END AS trend_direction
    FROM content_daily_labeled
)
SELECT
    trend_direction,
    CASE WHEN trend_direction = 'down' THEN 1 ELSE 0 END AS leaked_decline_feature
FROM labeled
LIMIT 20
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┬────────────────────────┐
│ trend_direction │ leaked_decline_feature │
│     varchar     │         int32          │
├─────────────────┼────────────────────────┤
│ NULL            │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up      

In [ ]:
con.sql("""
WITH labeled AS (
    SELECT
        *,
        CASE
            WHEN prior_obs_count >= 1 AND gsc_clicks < trailing_avg_clicks_7d THEN 'down'
            WHEN prior_obs_count >= 1 THEN 'up'
            ELSE NULL
        END AS trend_direction
    FROM content_daily_labeled
),
checked AS (
    SELECT
        trend_direction,
        CASE WHEN trend_direction = 'down' THEN 1 ELSE 0 END AS leaked_decline_feature
    FROM labeled
)
SELECT
    COUNT(*) AS rows,
    SUM(CASE
        WHEN leaked_decline_feature = CASE WHEN trend_direction = 'down' THEN 1 ELSE 0 END
        THEN 1 ELSE 0
    END) AS matching_rows
FROM checked
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬───────────────┐
│  rows   │ matching_rows │
│  int64  │    int128     │
├─────────┼───────────────┤
│ 9841378 │       9841378 │
└─────────┴───────────────┘

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Uman-66/Flyrank-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
!pip -q install duckdb


In [ ]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN is None:
    raise ValueError("HF_TOKEN was not found in Colab Secrets.")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

print("Hugging Face connection ready.")

Hugging Face connection ready.


In [ ]:
DAILY_PATH = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/**/*.parquet"
)

con.sql(f"""
SELECT COUNT(*) AS total_rows
FROM read_parquet(
    '{DAILY_PATH}',
    hive_partitioning=true
)
""")

MONTH = "2026-03"

Section 1 — Unit of analysis + time window

Markdown:

One row represents the daily observed performance of one content item, for one client, on one report date. The grain is report_date + client_hash_id + content_hash_id.

Time window: month = '2026-03' (March 2026), a mid-panel month. June 2026 is the sealed final month and is not used to define label logic.

Target/label: the schema has no pre-built trend column, so the label is derived: for each (client, content) pair, trend_direction = 'down' if that row's gsc_clicks is below the pair's own trailing 7-observation average clicks (computed from strictly prior rows only), else 'up'. This is a self-referential, backward-looking comparison — no future data is used to build the label.

Purpose: refresh prioritization / content opportunity scoring. This is an observed/directional ranking task, not a causal claim.

In [ ]:
con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT report_date) AS distinct_report_dates,
    MIN(report_date) AS first_report_date,
    MAX(report_date) AS last_report_date
FROM read_parquet(
    '{DAILY_PATH}',
    hive_partitioning=true
)
WHERE month = '{MONTH}'
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────┬───────────────────────┬───────────────────┬──────────────────┐
│ row_count │ distinct_report_dates │ first_report_date │ last_report_date │
│   int64   │         int64         │       date        │       date       │
├───────────┼───────────────────────┼───────────────────┼──────────────────┤
│   9841378 │                    31 │ 2026-03-01        │ 2026-03-31       │
└───────────┴───────────────────────┴───────────────────┴──────────────────┘

In [ ]:
con.sql(f"""
CREATE OR REPLACE VIEW content_daily_labeled AS
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    client_has_gsc,
    gsc_data_available,
    gsc_clicks,
    gsc_impressions,
    gsc_avg_position,
    ga4_engaged_sessions,

    COUNT(gsc_clicks) OVER (
        PARTITION BY client_hash_id, content_hash_id
        ORDER BY report_date
        ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
    ) AS prior_obs_count,

    AVG(gsc_clicks) OVER (
        PARTITION BY client_hash_id, content_hash_id
        ORDER BY report_date
        ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
    ) AS trailing_avg_clicks_7d

FROM read_parquet(
    '{DAILY_PATH}',
    hive_partitioning=true
)
WHERE month = '{MONTH}'
""")

print("view created")

view created


In [ ]:
con.sql("""
SELECT
    *,
    CASE
        WHEN prior_obs_count >= 1 AND gsc_clicks < trailing_avg_clicks_7d THEN 'down'
        WHEN prior_obs_count >= 1 THEN 'up'
        ELSE NULL
    END AS trend_direction
FROM content_daily_labeled
LIMIT 5
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────────┬────────────┬─────────────────┬──────────────────┬──────────────────────┬─────────────────┬────────────────────────┬─────────────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ gsc_data_available │ gsc_clicks │ gsc_impressions │ gsc_avg_position │ ga4_engaged_sessions │ prior_obs_count │ trailing_avg_clicks_7d │ trend_direction │
│    date     │         varchar         │         varchar          │    boolean     │      boolean       │   int64    │      int64      │      double      │        int64         │      int64      │         double         │     varchar     │
├─────────────┼─────────────────────────┼──────────────────────────┼────────────────┼────────────────────┼────────────┼─────────────────┼──────────────────┼──────────────────────┼─────────────────┼────────────────────────┼─────────────────┤
│ 2026-03-01  │ client_0797ff3a1fc9a

Section 2 — Fields: feature / label / context / excluded

Markdown:

Label: trend_direction (derived, see Section 1) — down means the day's clicks fell below the content's own recent trailing average. This is the outcome and is never used as an input feature.

Context: client_hash_id, content_hash_id, report_date, month.

Features (five):

trailing_avg_clicks_7d — knowable at the decision moment because it's computed only from the 7 prior observations of the same content, excluding the current day.
gsc_avg_position — knowable at the decision moment because it's the search ranking position observed as of that day's report, not a future value.
gsc_impressions — knowable at the decision moment because it's the same-day observed impression count, already measured by the time the report exists.
ga4_engaged_sessions — knowable at the decision moment because it's the same-day observed engagement count.
client_has_gsc — knowable at the decision moment because it's a static client attribute, true regardless of date.

Excluded: trend_direction (the label itself), prior_obs_count (a bookkeeping column, not a real signal), and anything computed from a period after the current report_date — none of the five features above reach into the future, but any such column would be excluded for that reason.

In [ ]:
con.sql("""
SELECT
    CASE
        WHEN prior_obs_count >= 1 AND gsc_clicks < trailing_avg_clicks_7d THEN 'down'
        WHEN prior_obs_count >= 1 THEN 'up'
        ELSE NULL
    END AS trend_direction,
    COUNT(*) AS rows
FROM content_daily_labeled
GROUP BY 1
ORDER BY rows DESC
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┬─────────┐
│ trend_direction │  rows   │
│     varchar     │  int64  │
├─────────────────┼─────────┤
│ up              │ 8690804 │
│ down            │  819137 │
│ NULL            │  331437 │
└─────────────────┴─────────┘

In [ ]:
con.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE client_hash_id IS NULL) AS missing_client_hash_id,
    COUNT(*) FILTER (WHERE content_hash_id IS NULL) AS missing_content_hash_id,
    COUNT(*) FILTER (WHERE report_date IS NULL) AS missing_report_date,
    COUNT(*) FILTER (WHERE prior_obs_count = 0) AS rows_with_no_prior_history
FROM content_daily_labeled
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────────┬─────────────────────────┬─────────────────────┬────────────────────────────┐
│ total_rows │ missing_client_hash_id │ missing_content_hash_id │ missing_report_date │ rows_with_no_prior_history │
│   int64    │         int64          │          int64          │        int64        │           int64            │
├────────────┼────────────────────────┼─────────────────────────┼─────────────────────┼────────────────────────────┤
│    9841378 │                      0 │                       0 │                   0 │                     331437 │
└────────────┴────────────────────────┴─────────────────────────┴─────────────────────┴────────────────────────────┘

Section 3 — Verify it with queries

Markdown — grain verification:

Expected grain: one row per report_date + client_hash_id + content_hash_id.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS rows_per_grain
FROM read_parquet(
    '{DAILY_PATH}',
    hive_partitioning=true
)
WHERE month = '{MONTH}'
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
ORDER BY rows_per_grain DESC
LIMIT 20
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬────────────────┐
│ report_date │ client_hash_id │ content_hash_id │ rows_per_grain │
│    date     │    varchar     │     varchar     │     int64      │
├─────────────┴────────────────┴─────────────────┴────────────────┤
│                             0 rows                              │
└─────────────────────────────────────────────────────────────────┘

In [ ]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT
        CAST(report_date AS VARCHAR) || '|' ||
        CAST(client_hash_id AS VARCHAR) || '|' ||
        CAST(content_hash_id AS VARCHAR)
    ) AS unique_grain_keys
FROM read_parquet(
    '{DAILY_PATH}',
    hive_partitioning=true
)
WHERE month = '{MONTH}'
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬───────────────────┐
│ total_rows │ unique_grain_keys │
│   int64    │       int64       │
├────────────┼───────────────────┤
│    9841378 │           9841378 │
└────────────┴───────────────────┘

In [ ]:
con.sql(f"""
SELECT
    COUNT(*) AS all_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS rows_surviving_is_true
FROM read_parquet(
    '{DAILY_PATH}',
    hive_partitioning=true
)
WHERE month = '{MONTH}'
""")

┌──────────┬────────────────────────┐
│ all_rows │ rows_surviving_is_true │
│  int64   │         int64          │
├──────────┼────────────────────────┤
│  9841378 │                3611061 │
└──────────┴────────────────────────┘

In [ ]:
con.sql("""
WITH labeled AS (
    SELECT
        *,
        CASE
            WHEN prior_obs_count >= 1 AND gsc_clicks < trailing_avg_clicks_7d THEN 'down'
            WHEN prior_obs_count >= 1 THEN 'up'
            ELSE NULL
        END AS trend_direction
    FROM content_daily_labeled
)
SELECT
    trend_direction,
    CASE WHEN trend_direction = 'down' THEN 1 ELSE 0 END AS leaked_decline_feature
FROM labeled
LIMIT 20
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┬────────────────────────┐
│ trend_direction │ leaked_decline_feature │
│     varchar     │         int32          │
├─────────────────┼────────────────────────┤
│ NULL            │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up              │                      0 │
│ up      

In [ ]:
con.sql("""
WITH labeled AS (
    SELECT
        *,
        CASE
            WHEN prior_obs_count >= 1 AND gsc_clicks < trailing_avg_clicks_7d THEN 'down'
            WHEN prior_obs_count >= 1 THEN 'up'
            ELSE NULL
        END AS trend_direction
    FROM content_daily_labeled
),
checked AS (
    SELECT
        trend_direction,
        CASE WHEN trend_direction = 'down' THEN 1 ELSE 0 END AS leaked_decline_feature
    FROM labeled
)
SELECT
    COUNT(*) AS rows,
    SUM(CASE
        WHEN leaked_decline_feature = CASE WHEN trend_direction = 'down' THEN 1 ELSE 0 END
        THEN 1 ELSE 0
    END) AS matching_rows
FROM checked
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬───────────────┐
│  rows   │ matching_rows │
│  int64  │    int128     │
├─────────┼───────────────┤
│ 9841378 │       9841378 │
└─────────┴───────────────┘

Section 4 — Data limits

Markdown:

Unbalanced history. Content with few prior observations gets a noisier trailing_avg_clicks_7d baseline than long-running content — check rows_with_no_prior_history above.

Observational, not causal. A down label means clicks fell relative to the content's own recent baseline — it does not prove refreshing the page would reverse that.

GSC-only early rows. Some rows may have gsc_data_available = FALSE, meaning search-console-based label/features aren't trustworthy for that row (see the IS TRUE check above).

Window overlap / early-March edge effect. Because the label window only looks at prior rows within March, the first few days of March have a smaller trailing window than later days — a boundary effect, not a leak, but worth naming.

Sealed final month. June 2026 is not used to define label logic here.

In [41]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4: data-limit checks

con.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE prior_obs_count = 0
    ) AS rows_with_no_prior_history,
    COUNT(*) FILTER (
        WHERE prior_obs_count BETWEEN 1 AND 6
    ) AS rows_with_less_than_7_prior_observations,
    COUNT(*) FILTER (
        WHERE prior_obs_count >= 7
    ) AS rows_with_full_7_observation_history
FROM content_daily_labeled
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────────────┬──────────────────────────────────────────┬──────────────────────────────────────┐
│ total_rows │ rows_with_no_prior_history │ rows_with_less_than_7_prior_observations │ rows_with_full_7_observation_history │
│   int64    │           int64            │                  int64                   │                int64                 │
├────────────┼────────────────────────────┼──────────────────────────────────────────┼──────────────────────────────────────┤
│    9841378 │                     331437 │                                  1961457 │                              7548484 │
└────────────┴────────────────────────────┴──────────────────────────────────────────┴──────────────────────────────────────┘

In [42]:
# Check GSC availability

con.sql("""
SELECT
    gsc_data_available,
    COUNT(*) AS rows
FROM content_daily_labeled
GROUP BY gsc_data_available
ORDER BY gsc_data_available
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────┬─────────┐
│ gsc_data_available │  rows   │
│      boolean       │  int64  │
├────────────────────┼─────────┤
│ false              │ 6230317 │
│ true               │ 3611061 │
└────────────────────┴─────────┘

In [43]:
# Check how much prior history is available across March observations

con.sql("""
SELECT
    prior_obs_count,
    COUNT(*) AS rows
FROM content_daily_labeled
GROUP BY prior_obs_count
ORDER BY prior_obs_count
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┬─────────┐
│ prior_obs_count │  rows   │
│      int64      │  int64  │
├─────────────────┼─────────┤
│               0 │  331437 │
│               1 │  331230 │
│               2 │  326193 │
│               3 │  326193 │
│               4 │  326193 │
│               5 │  326040 │
│               6 │  325608 │
│               7 │ 7548484 │
└─────────────────┴─────────┘

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [39]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.